# Part 3: CI/CD & Production Workflows

In production environments, you rarely want to hardcode comparison rules inside your application logic. This notebook demonstrates how to decouple **Policy** (your comparison rules) from **Execution** (your pipeline code) using Veridelta's YAML configuration.

### 1. The Production Pattern
The standard pattern for Veridelta in a CI/CD pipeline is:
1. **Store** your `veridelta.yaml` in your git repository.
2. **Execute** a generic script that loads the config and runs the diff.
3. **Update** comparison tolerances via Pull Requests without touching the core code.

In [ ]:
from pathlib import Path
import polars as pl
from veridelta import load_config, DataIngestor, DiffEngine

# Mock production data
src = pl.DataFrame({"id": [1], "val": [10.0]})
tgt = pl.DataFrame({"id": [1], "val": [10.02]})
src.write_parquet("prod_source.parquet")
tgt.write_parquet("prod_target.parquet")

# The 'Policy' file that lives in Git
yaml_content = """
primary_keys: ["id"]
default_absolute_tolerance: 0.05
source:
  path: "prod_source.parquet"
  format: "parquet"
target:
  path: "prod_target.parquet"
  format: "parquet"
"""
Path("prod_config.yaml").write_text(yaml_content)
print("Production config and data ready.")

### 2. Generic Execution Script
In your CI/CD runner (like GitHub Actions), you would run a script similar to the one below. Notice that this script doesn't know *anything* about the columns or tolerances—it just follows the instructions in the YAML.

In [ ]:
def run_pipeline_validation(config_path: str):
    # Load and validate the external policy
    config = load_config(config_path)
    
    # Align and Ingest
    ingestor = DataIngestor(config)
    source_df, target_df = ingestor.get_dataframes()
    
    # Execute Diff
    engine = DiffEngine(config, source_df, target_df)
    summary = engine.run()
    
    if summary.is_match:
        print("✅ CI/CD Validation Passed")
    else:
        print(f"❌ Regression Detected: {summary.changed_count} differences found.")
        # In a real CI, you might raise an error here to fail the build

run_pipeline_validation("prod_config.yaml")

### Pro Tip: Environment Overrides
When moving from `Staging` to `Production`, you can use the `SourceConfig.options` to pass dynamic credentials or path overrides, keeping your Veridelta rules consistent across every environment.